Check how many SNPs in seed/simulated BAM are from the input list of phased SNPs.

The SNPs in BAM are called by cellsnp-lite mode 2a with parameters `--minCOUNT 11` and `--minMAF 0.1` (also with `-f` option).

In [1]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [2]:
from sccnasim.xlib.xvcf import vcf_load
from sccnasim.xlib.xrange import format_chrom

In [3]:
phased_snp_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/base/data/snp/HCC3N.phased.vcf.gz"

seed_snp_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/cellsnp_mode2a/cellSNP.base.vcf.gz"
seed_snp_xf_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/xf_tag/cellsnp_mode2a/cellSNP.base.vcf.gz"
rs_snp_xf_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/cellsnp_mode2a/cellSNP.base.vcf.gz"
scrs_snp_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/cellsnp_mode2a/cellSNP.base.vcf.gz"
scrs_snp_xf_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/align/cellranger/xf_tag/cellsnp_mode2a/cellSNP.base.vcf.gz"

In [4]:
out_fn = "cellsnp_mode2a.stat.tsv"

In [5]:
tmp_fn = out_fn + ".tmp"
fp = open(tmp_fn, "w")
fp.write("\t".join([
    "group", "n_cell", "n_phase", "n_call", 
    "n_shared_chrom_pos", "n_shared_chrom_pos_ref_alt"]) + "\n")

for call_snp_fn, label, n_cell in zip(
    [seed_snp_fn, seed_snp_xf_fn, rs_snp_xf_fn, scrs_snp_fn, scrs_snp_xf_fn],
    ["seed", "seed_xf", "rs_xf", "scrs", "scrs_xf"],
    [600, 600, 1200, 1200, 1200]
):
    print("processing '%s' ..." % label)
    df_phase, header_phase = vcf_load(phased_snp_fn)
    df_phase["chrom_pos"] = df_phase["CHROM"].astype(str).map(format_chrom) + "_" + df_phase["POS"].astype(str)
    df_phase["chrom_pos_ref_alt"] = df_phase["chrom_pos"] + "_" + df_phase["REF"] + "_" + df_phase["ALT"]
    
    df_call, header_call = vcf_load(call_snp_fn)
    df_call["chrom_pos"] = df_call["CHROM"].astype(str).map(format_chrom) + "_" + df_call["POS"].astype(str)
    df_call["chrom_pos_ref_alt"] = df_call["chrom_pos"] + "_" + df_call["REF"] + "_" + df_call["ALT"]

    fp.write("\t".join([
            label,
            str(n_cell),
            str(df_phase.shape[0]),
            str(df_call.shape[0]),
            str(np.sum(df_call["chrom_pos"].isin(df_phase["chrom_pos"]))),
            str(np.sum(df_call["chrom_pos_ref_alt"].isin(df_phase["chrom_pos_ref_alt"])))
        ]) + "\n"
    )

fp.close()

processing 'seed' ...
processing 'seed_xf' ...
processing 'rs_xf' ...
processing 'scrs' ...
processing 'scrs_xf' ...


In [6]:
df = pd.read_csv(tmp_fn, sep = '\t')
df

,group,n_cell,n_phase,n_call,n_shared_chrom_pos,n_shared_chrom_pos_ref_alt
0,seed,600,8866,22679,1972,1969
1,seed_xf,600,8866,15295,1478,1477
2,rs_xf,1200,8866,2045,2045,2045
3,scrs,1200,8866,6244,2,1
4,scrs_xf,1200,8866,2462,1,1


In [7]:
df.index = df['group']
df.index.name = None
del df['group']
df = df.transpose()
df

,seed,seed_xf,rs_xf,scrs,scrs_xf
n_cell,600,600,1200,1200,1200
n_phase,8866,8866,8866,8866,8866
n_call,22679,15295,2045,6244,2462
n_shared_chrom_pos,1972,1478,2045,2,1
n_shared_chrom_pos_ref_alt,1969,1477,2045,1,1


In [8]:
df.to_csv(out_fn, sep = '\t', header = True, index = True)